# Bronze ingestion framework

In [0]:
from pyspark.sql.functions import *


In [0]:
def ingest_to_bronze(table_name, file_name):

    raw_base_path = "/Volumes/e_comm_databricks/default/olist_raw_volume/"

    df = (
        spark.read \
            .format("csv")\
            .option("header", True)\
            .option("inferSchema", True)\
            .load(raw_base_path + file_name)
    )
    df_with_meta = (
    df
        .withColumn("_ingest_timestamp", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingest_date", to_date(current_timestamp()))
)


    (
        df_with_meta
        .write\
            .mode("append")\
            .format("delta")\
           .saveAsTable(f"e_comm_databricks.bronze.{table_name}")
    )

In [0]:
ingest_to_bronze("olist_orders","olist_orders_dataset.csv")

ingest_to_bronze("olist_products","olist_products_dataset.csv")

ingest_to_bronze("olist_customers","olist_customers_dataset.csv")

ingest_to_bronze("olist_geolocation","olist_geolocation_dataset.csv")

ingest_to_bronze("olist_order_items","olist_order_items_dataset.csv")

ingest_to_bronze("olist_sellers","olist_sellers_dataset.csv")

ingest_to_bronze("olist_order_payments","olist_order_payments_dataset.csv")

ingest_to_bronze("olist_order_reviews","olist_order_reviews_dataset.csv")
   

In [0]:
bronze_tables = {
    "olist_orders": "olist_orders_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_customers": "olist_customers_dataset.csv",
    "olist_geolocation": "olist_geolocation_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv"
}


In [0]:
for table_name, file_name in bronze_tables.items():
    print(f"Ingesting {table_name}")
    ingest_to_bronze(table_name, file_name)
